# Kaggriculture Adaptive Bandit (private) - v63.17_rl_f898

v63.17 (F6: v63.16 + sale cash floor + 10 world routes) validation (not submitted)

`submission.tar.gz` = `main.py` stdio bridge + static Linux `agent-stdio` (Rust port of v61.1, lever profile None) + route tables + profile table + Python v61.1 fallback.

4,290,013 bytes, sha256 `2ac28fc7` (binary `feef4479fc20`); read from the private dataset `debmalya84/kaggriculture-rl-payload`, folder `None`.

In [ ]:
import glob, hashlib, json, os, shutil, tarfile, zipfile
TAR_SHA256 = '2ac28fc74973a46188335c5f210e9043a9c93a8dc2967c76fcda38344ee57245'
TAR_SIZE = 4290013
SUBDIR = None
PAYLOAD = 'v63_17_rl_f898.submission.tar.gz.bin'


## payload

In [ ]:
if PAYLOAD:  # a flat file in the dataset
    hits = [p for p in glob.glob('/kaggle/input/**/*.bin', recursive=True) if os.path.basename(p) == PAYLOAD]
else:
    hits = [p for p in glob.glob('/kaggle/input/**/submission.tar.gz.bin', recursive=True) if f'/{SUBDIR}/' in p]
if not hits:  # the folder may be mounted as a zip
    for z in glob.glob(f'/kaggle/input/**/{SUBDIR}.zip', recursive=True):
        zipfile.ZipFile(z).extractall('/tmp/payload')
    hits = glob.glob('/tmp/payload/**/submission.tar.gz.bin', recursive=True)
print('payload', hits)
assert len(hits) == 1, hits
data = open(hits[0], 'rb').read()
assert len(data) == TAR_SIZE, (len(data), TAR_SIZE)
assert hashlib.sha256(data).hexdigest() == TAR_SHA256
open('/kaggle/working/submission.tar.gz', 'wb').write(data)
print('wrote submission.tar.gz', len(data), 'bytes; sha', TAR_SHA256)


## validation episode: unpack and play a full self-play episode on the official engine (DONE/DONE, Rust bridge every turn)

In [ ]:
import io, contextlib
AG = '/kaggle/working/_agent'
shutil.rmtree(AG, ignore_errors=True); os.makedirs(AG)
with tarfile.open('/kaggle/working/submission.tar.gz') as t:
    t.extractall(AG)
print(sorted(os.listdir(AG)))
from kaggle_environments import make
env = make('kaggriculture', debug=True)
err = io.StringIO()
with contextlib.redirect_stderr(err):
    env.run([os.path.join(AG, 'main.py'), os.path.join(AG, 'main.py')])
st = [s.get('status') for s in env.steps[-1]]
rw = [s.get('reward') for s in env.steps[-1]]
beacons = [l for l in err.getvalue().splitlines() if l.startswith('RUSTV61 end')]
print('statuses', st); print('rewards ', rw)
for b in beacons[-2:]:
    print(b)
assert all(s == 'DONE' for s in st), st
stats = [json.loads(b.split(' ', 2)[2]) for b in beacons[-2:]]
if stats:
    # the Rust binary must have played every turn of both seats, well inside actTimeout (1 s)
    assert all(s['fallback'] == 0 and s['timeouts'] == 0 and s['bad_reply'] == 0 for s in stats), stats
    assert all(s['turns'] >= 719 and s['bridge'] == s['turns'] for s in stats), stats
    assert all(s['worst_ms'] < 1000.0 for s in stats), [s['worst_ms'] for s in stats]
    print('worst turn ms', [round(s['worst_ms'], 1) for s in stats], 'build', stats[0].get('build'))
else:
    print('WARNING: no RUSTV61 beacons captured; the DONE check is still enforced')
shutil.rmtree(AG, ignore_errors=True)
print('VALIDATION PASS (self-play episode DONE/DONE, beacons %d); outputs: %s' % (len(stats), os.listdir('/kaggle/working')))
